# Ruteo de distribución urbana — Valle de Aburrá

**Problema:** cada día una flota heterogénea sale del Centro de Distribución (nodo 0), entrega pedidos
en puntos de venta y regresa al mismo depot. Se busca el plan de **mínimo costo operativo**
(km × costo/km del vehículo asignado) que respete capacidad, jornada y flota disponible.
Es un **HFVRP** (*Heterogeneous Fleet Vehicle Routing Problem*) con límite de jornada.

Este notebook **no implementa lógica propia**: importa `ruteo.py`, el mismo módulo que usa la aplicación web.

| Paso | Contenido |
|---|---|
| 1 | Parámetros de operación |
| 2 | Lectura de datos |
| 3 | Validación |
| 4 | Modelo matemático y método |
| 5 | Un día paso a paso (Lunes) |
| 6 | Todos los días |
| 7 | Detalle de rutas |
| 8 | Verificación de restricciones |
| 9 | Días infactibles: diagnóstico |
| 10 | Cómo iniciar la aplicación |

In [1]:
import json
from pathlib import Path

import pandas as pd

import ruteo

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
ARCHIVO = Path(ruteo.CONFIG["archivo_base"])
print("Archivo:", ARCHIVO.name)

Archivo: Reto_VRP_Medellin_Datos.xlsx


## 1. Parámetros de operación

Todos los parámetros viven en un único bloque, `ruteo.CONFIG`, compartido con la aplicación.
Para cambiar uno basta `ruteo.parametros_por_defecto(tiempo_limite_s=60)`.

In [2]:
pd.DataFrame(
    [(k, v) for k, v in ruteo.CONFIG.items() if k not in ("hojas", "archivo_base")],
    columns=["Parámetro", "Valor"],
)

,Parámetro,Valor
0,velocidad_kmh,24.0
1,descarga_min,30.0
2,inicio_jornada,07:00
3,jornada_max_min,480.0
4,nodo_depot,0
5,tiempo_limite_s,30.0
6,estrategias_iniciales,"[PATH_CHEAPEST_ARC, PARALLEL_CHEAPEST_INSERTIO..."
7,metaheuristica,GUIDED_LOCAL_SEARCH
8,fraccion_fase_inicial,0.25
9,tiempo_empaque_s,10.0


## 2. Lectura de datos

`preparar_datos` lee las cuatro hojas y las convierte a estructuras limpias:

- **Clientes:** nodo, nombre, dirección y coordenadas. Las coordenadas solo se usan para el mapa.
- **Matriz_Distancias:** el número de nodo se extrae del rótulo `"N - Nombre"`. Fila = origen y columna = destino.
  Se usa **tal como viene**, asimétrica.
- **Vehiculos:** capacidad y costo vienen como texto (`"10 toneladas"`, `"4.200 COP/km"`). El punto es separador de miles.
- **Demanda:** los días son las columnas a la derecha de `Cliente`, cualesquiera que sean sus nombres.

In [3]:
datos, problemas = ruteo.preparar_datos(ARCHIVO, ARCHIVO.name)
pd.DataFrame(datos.clientes).head(8)

,nodo,nombre,direccion,lat,lon
0,0,La Macarena Medellín,"Cra. 63A #45a-48, Laureles, Medellín",6.251360,-75.580875
1,1,Éxito Mayorca,"Cl. 44 #50 Sur-45, Sabaneta",6.161091,-75.603690
2,2,D1 Sabaneta Parque,Sabaneta (centro),6.150180,-75.618947
3,3,Ara Sabaneta Holanda,"Holanda, Sabaneta",6.150873,-75.618292
4,4,Éxito Itagüí,"Cra. 50A #41-42, Itagüí",6.169084,-75.616378
5,5,D1 Itagüí Centro,Itagüí (centro),6.168291,-75.618796
6,6,Ara Ditaires,"Cl. 34B #53-57, Ditaires, Itagüí",6.169487,-75.628144
7,7,Éxito Viva Envigado,"Cra. 52 #2-38, Guayabal, Medellín",6.209075,-75.583073


In [4]:
m = datos.matriz
print(f"Matriz {m.km.shape[0]} × {m.km.shape[1]} km")
asimetricos = sum(1 for i in range(len(m.nodos)) for j in range(i + 1, len(m.nodos)) if abs(m.km[i, j] - m.km[j, i]) > 1e-9)
total_pares = len(m.nodos) * (len(m.nodos) - 1) // 2
print(f"Pares asimétricos: {asimetricos} de {total_pares}")
print(f"Ejemplo: 0→1 = {m.distancia(0, 1)} km, 1→0 = {m.distancia(1, 0)} km")
pd.DataFrame(m.km[:6, :6], index=m.nodos[:6], columns=m.nodos[:6])

Matriz 31 × 31 km
Pares asimétricos: 445 de 465
Ejemplo: 0→1 = 12.1 km, 1→0 = 13.8 km


,0,1,2,3,4,5
0,0.0,12.1,13.6,13.5,11.8,12.2
1,13.8,0.0,2.5,2.4,2.6,2.5
2,16.1,3.4,0.0,0.4,4.3,4.2
3,16.2,3.5,0.1,0.0,4.4,4.3
4,16.3,3.2,3.6,3.5,0.0,0.5
5,16.0,3.1,3.2,3.1,0.7,0.0


In [5]:
pd.DataFrame([vars(t) for t in datos.flota])

,tipo,codigo,cantidad,capacidad_t,costo_km
0,Camión Tipo A,A,1,10.0,4200.0
1,Camión Tipo B,B,2,6.0,3700.0
2,Camión Tipo C,C,2,3.0,3300.0


In [6]:
demanda = pd.DataFrame({dia: pd.Series(datos.demanda[dia]) if datos.demanda[dia] is not None else pd.Series(dtype=float)
                        for dia in datos.dias}).reindex([c["nodo"] for c in datos.clientes if c["nodo"] != 0])
demanda.index.name = "Nodo"
demanda.fillna(0).sum().to_frame("Toneladas").T

,Lunes,Martes,Miércoles,Jueves,Viernes,Sábado
Toneladas,27.5,27.1,14.9,24.8,0.0,0.0


## 3. Validación

Antes de optimizar se revisan las hojas y columnas faltantes, los nodos de Demanda que no existen en Clientes o en la matriz,
una matriz no cuadrada o con vacíos, las demandas negativas o no numéricas, las coordenadas fuera de rango
y los pedidos mayores que la capacidad del vehículo más grande.

Un día con la columna vacía es un aviso, no un error. Ese día se omite.

In [7]:
pd.DataFrame(problemas)

,nivel,hoja,columna,nodo,problema
0,aviso,Demanda,Viernes,,"Columna vacía: día sin datos, no se calcula."
1,aviso,Demanda,Sábado,,"Columna vacía: día sin datos, no se calcula."


Ejemplo con un archivo con errores (columna faltante, texto en una demanda y un nodo inexistente):

In [8]:
_, errores = ruteo.preparar_datos("datos/pruebas/e_con_errores.xlsx")
pd.DataFrame(errores)[["hoja", "columna", "nodo", "problema"]]

,hoja,columna,nodo,problema
0,Clientes,Latitud,,Falta la columna «Latitud».
1,Demanda,Martes,4,Demanda no numérica: «uno coma dos» no es un n...
2,Demanda,Nodo,45,El nodo no existe en la hoja Clientes.


Resumen previo, antes de resolver: días con datos, clientes con pedido, toneladas y su relación con la capacidad de la flota.

In [9]:
previo = ruteo.resumen_previo(datos)
print(f"Capacidad de la flota: {previo['capacidad_flota_t']} t en {previo['vehiculos_disponibles']} vehículos")
pd.DataFrame(previo["dias"])

Capacidad de la flota: 28.0 t en 5 vehículos


,dia,con_datos,clientes,toneladas,excede_capacidad,uso_pct
0,Lunes,True,30.0,27.5,False,98.2
1,Martes,True,30.0,27.1,False,96.8
2,Miércoles,True,22.0,14.9,False,53.2
3,Jueves,True,30.0,24.8,False,88.6
4,Viernes,False,NaN,NaN,NaN,NaN
5,Sábado,False,NaN,NaN,NaN,NaN


## 4. Modelo y método

**Conjuntos:** $N$ = clientes con demanda > 0 del día; $V$ = vehículos físicos (A1, B1, B2, C1, C2), cada uno con capacidad $Q_v$ y costo $c_v$ (COP/km).

**Objetivo:** $\min \sum_{v}\sum_{(i,j)} c_v \, d_{ij} \, x_{ijv}$, con $d_{ij}$ tomado de la matriz asimétrica en el sentido del tramo.
Un vehículo sin clientes no sale y no cuesta nada.

**Restricciones:**
1. Cada cliente se visita exactamente una vez, por un solo vehículo. Los pedidos no se dividen.
2. Cada ruta inicia y termina en el depot.
3. Carga de la ruta ≤ $Q_v$.
4. Tiempo de la ruta, contando viaje a 24 km/h, 30 min de descarga por cliente y regreso, ≤ 480 min.
5. Solo se usan los vehículos disponibles. El modelo decide cuántos salen.

**Implementación con OR-Tools Routing:**
- Escalas enteras: metros, kilogramos y segundos. La demanda y el tiempo se redondean hacia arriba y la capacidad hacia abajo, así el modelo entero nunca es más optimista que la realidad.
- Costo de arco distinto por vehículo: `SetArcCostEvaluatorOfVehicle` con metros × COP/km.
- Dimensión `Carga` con capacidad por vehículo.
- Dimensión `Tiempo`, cuyo tránsito i→j es la descarga en i más el viaje i→j, con tope de 480 min.

**Búsqueda, en fases:**
1. Se prueban cinco estrategias de solución inicial y se toma la más barata.
2. Desde ella, *Guided Local Search* mejora el plan hasta el tiempo límite de 30 s por día.

**Respaldos si ninguna estrategia inicial encuentra un plan factible** (la capacidad está muy ajustada: 27,5 t contra 28 t):
- Modelo con penalización alta por cliente omitido. Siempre tiene punto de partida, y la metaheurística reduce las omisiones a cero.
- Asignación previa tipo *bin packing* con CP-SAT (capacidad + cota inferior de jornada), secuenciación por vehículo y cortes cuando un grupo no cabe en la jornada.

**Diagnóstico de días infactibles**, con cifras:
- Demanda total mayor que la capacidad de la flota.
- Pedidos que no caben sin dividirse, demostrado con CP-SAT.
- Jornada insuficiente: un cliente inalcanzable, o un reparto imposible con la cota de tiempo.

## 5. Un día paso a paso: Lunes

In [10]:
params = ruteo.parametros_por_defecto()
inst = ruteo.preparar_instancia(datos.demanda["Lunes"], datos.matriz, datos.flota, params)
print("Nodos del día:", len(inst["nodos"]) - 1, "clientes + depot")
print("Vehículos:", [v["id"] for v in inst["vehiculos"]])
print("Capacidades (kg):", inst["capacidad_kg"], "· Demanda total (kg):", sum(inst["demanda_kg"]))
print("Costo por vehículo (COP/km × 100):", inst["costo_x100"])
print("Tope de jornada (s):", inst["limite_s"])

Nodos del día: 30 clientes + depot
Vehículos: ['A1', 'B1', 'B2', 'C1', 'C2']
Capacidades (kg): [10000, 6000, 6000, 3000, 3000] · Demanda total (kg): 27500
Costo por vehículo (COP/km × 100): [420000, 370000, 370000, 330000, 330000]
Tope de jornada (s): 28800


Fase 1: soluciones iniciales. Con 27,5 t contra 28 t, varias estrategias no encuentran un plan factible.

In [11]:
iniciales = ruteo.fase_soluciones_iniciales(inst, params, params["tiempo_limite_s"] * params["fraccion_fase_inicial"])
factibles = {e: costo / 1e5 for costo, e, _ in iniciales}
pd.DataFrame([{"Estrategia": e, "Factible": e in factibles, "Costo inicial (COP)": factibles.get(e)}
              for e in params["estrategias_iniciales"]])

,Estrategia,Factible,Costo inicial (COP)
0,PATH_CHEAPEST_ARC,False,NaN
1,PARALLEL_CHEAPEST_INSERTION,False,NaN
2,SAVINGS,False,NaN
3,CHRISTOFIDES,False,NaN
4,LOCAL_CHEAPEST_INSERTION,True,664850.0


Respaldo con bin packing. Se ejecuta aquí solo para mostrarlo: en la corrida normal solo se usa si la fase 1 no encuentra nada.

In [12]:
rutas_respaldo = ruteo.respaldo_empaque(inst, 10, params)
print("Respaldo factible:", rutas_respaldo is not None,
      "· costo de partida:", ruteo.formato_cop(ruteo._costo_entero(inst, rutas_respaldo) / 1e5) if rutas_respaldo else "-")

Respaldo factible: True · costo de partida: $ 715.100


Solución completa del Lunes con `resolver_ruteo`, la misma función que usa la aplicación:

In [13]:
sol_lunes = ruteo.resolver_ruteo(datos.demanda["Lunes"], datos.matriz, datos.flota, params)
print(sol_lunes["estado"], sol_lunes["calculo"])
for r in sol_lunes["rutas"]:
    print(f"{r['vehiculo']}: 0 → {' → '.join(map(str, r['secuencia']))} → 0")

resuelto {'estrategia': 'LOCAL_CHEAPEST_INSERTION', 'uso_respaldo': False, 'estrategias_factibles': ['LOCAL_CHEAPEST_INSERTION'], 'segundos': 30.0}
A1: 0 → 7 → 4 → 6 → 5 → 3 → 2 → 1 → 9 → 8 → 11 → 12 → 0
B1: 0 → 16 → 15 → 10 → 19 → 24 → 17 → 0
B2: 0 → 30 → 27 → 29 → 28 → 26 → 25 → 18 → 0
C1: 0 → 14 → 21 → 13 → 0
C2: 0 → 20 → 23 → 22 → 0


## 6. Todos los días

`resolver_archivo` es la función de alto nivel. Valida el archivo, resuelve los días con datos en paralelo,
calcula indicadores y verifica. Devuelve una estructura serializable a JSON, la misma que consume el tablero.

In [14]:
resultados = ruteo.resolver_archivo(ARCHIVO)
print(resultados["mensaje_final"])
ruteo.tabla_resumen(resultados)

4 día(s) resuelto(s) · 2 sin datos


,Día,Estado,Demanda (t),Costo (COP),COP por t,Km,Vehículos,Utilización (%),Regreso más tardío,Holgura (min)
0,Lunes,resuelto,27.5,493000.0,17927.0,129.7,5/5,98.2,14:15,45.0
1,Martes,resuelto,27.1,480460.0,17729.0,127.0,5/5,96.8,14:11,49.0
2,Miércoles,resuelto,14.9,383180.0,25717.0,101.9,3/5,67.7,13:42,78.0
3,Jueves,resuelto,24.8,464530.0,18731.0,121.0,4/5,99.2,14:46,14.0
4,Viernes,sin_datos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Sábado,sin_datos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Detalle de rutas por día

In [15]:
for d in resultados["dias"]:
    if d["estado"] != "resuelto":
        print(f"\n{d['dia']}: {d['estado'].replace('_', ' ')}")
        continue
    print(f"\n{d['dia']} — sin despachar: {', '.join(d['sin_despachar']) or 'ninguno'}")
    tabla = pd.DataFrame([{
        "Vehículo": r["vehiculo"], "Paradas": r["n_paradas"],
        "Carga (t)": round(r["carga_t"], 2), "Capacidad (t)": r["capacidad_t"], "% cap.": round(r["pct_capacidad"], 1),
        "Km": round(r["km"], 1), "Duración (min)": round(r["duracion_min"]), "% jornada": round(r["pct_jornada"], 1),
        "Regreso": r["hora_regreso"], "Costo (COP)": round(r["costo"]),
        "Secuencia": "0 → " + " → ".join(map(str, r["secuencia"])) + " → 0",
    } for r in d["rutas"]])
    display(tabla)


Lunes — sin despachar: ninguno


,Vehículo,Paradas,Carga (t),Capacidad (t),% cap.,Km,Duración (min),% jornada,Regreso,Costo (COP),Secuencia
0,A1,11,9.8,10.0,98.0,41.9,435,90.6,14:15,175980,0 → 7 → 4 → 6 → 5 → 3 → 2 → 1 → 9 → 8 → 11 → 1...
1,B1,6,6.0,6.0,100.0,17.8,224,46.8,10:44,65860,0 → 16 → 15 → 10 → 19 → 24 → 17 → 0
2,B2,7,6.0,6.0,100.0,50.4,336,70.0,12:36,186480,0 → 30 → 27 → 29 → 28 → 26 → 25 → 18 → 0
3,C1,3,3.0,3.0,100.0,8.9,112,23.4,08:52,29370,0 → 14 → 21 → 13 → 0
4,C2,3,2.7,3.0,90.0,10.7,117,24.3,08:57,35310,0 → 20 → 23 → 22 → 0



Martes — sin despachar: ninguno


,Vehículo,Paradas,Carga (t),Capacidad (t),% cap.,Km,Duración (min),% jornada,Regreso,Costo (COP),Secuencia
0,A1,11,10.0,10.0,100.0,40.4,431,89.8,14:11,169680,0 → 7 → 4 → 6 → 5 → 3 → 2 → 1 → 9 → 8 → 11 → 2...
1,B1,6,5.6,6.0,93.3,12.5,211,44.0,10:31,46250,0 → 14 → 12 → 10 → 15 → 16 → 17 → 0
2,B2,7,6.0,6.0,100.0,50.0,335,69.8,12:35,185000,0 → 13 → 30 → 27 → 29 → 28 → 26 → 25 → 0
3,C1,3,2.5,3.0,83.3,13.4,124,25.7,09:04,44220,0 → 19 → 24 → 18 → 0
4,C2,3,3.0,3.0,100.0,10.7,117,24.3,08:57,35310,0 → 20 → 23 → 22 → 0



Miércoles — sin despachar: C1, C2


,Vehículo,Paradas,Carga (t),Capacidad (t),% cap.,Km,Duración (min),% jornada,Regreso,Costo (COP),Secuencia
0,A1,5,3.2,10.0,32.0,12.3,181,37.7,10:01,51660,0 → 14 → 12 → 10 → 15 → 16 → 0
1,B1,9,6.0,6.0,100.0,52.6,402,83.6,13:42,194620,0 → 20 → 22 → 30 → 27 → 28 → 25 → 24 → 18 → 17...
2,B2,8,5.7,6.0,95.0,37.0,332,69.3,12:32,136900,0 → 7 → 4 → 5 → 2 → 1 → 8 → 11 → 21 → 0



Jueves — sin despachar: C1


,Vehículo,Paradas,Carga (t),Capacidad (t),% cap.,Km,Duración (min),% jornada,Regreso,Costo (COP),Secuencia
0,A1,12,10.0,10.0,100.0,42.3,466,97.0,14:46,177660,0 → 7 → 4 → 6 → 5 → 3 → 2 → 1 → 9 → 8 → 11 → 1...
1,B1,7,6.0,6.0,100.0,49.7,334,69.6,12:34,183890,0 → 21 → 30 → 27 → 29 → 28 → 26 → 25 → 0
2,B2,7,5.9,6.0,98.3,18.2,256,53.2,11:16,67340,0 → 16 → 15 → 10 → 19 → 24 → 18 → 17 → 0
3,C2,4,2.9,3.0,96.7,10.8,147,30.6,09:27,35640,0 → 20 → 22 → 23 → 13 → 0



Viernes: sin datos

Sábado: sin datos


Secuencia del vehículo más cargado del Lunes, con ETA y carga restante:

In [16]:
lunes = next(d for d in resultados["dias"] if d["dia"] == "Lunes")
pd.DataFrame(lunes["rutas"][0]["paradas"])[["orden", "nodo", "cliente", "eta", "salida", "entrega_t", "carga_restante_t", "km_tramo"]]

,orden,nodo,cliente,eta,salida,entrega_t,carga_restante_t,km_tramo
0,1,7,Éxito Viva Envigado,07:14,07:44,1.4,8.4,5.6
1,2,4,Éxito Itagüí,08:04,08:34,1.4,7.0,8.2
2,3,6,Ara Ditaires,08:39,09:09,0.5,6.5,1.8
3,4,5,D1 Itagüí Centro,09:12,09:42,0.9,5.6,1.4
4,5,3,Ara Sabaneta Holanda,09:50,10:20,0.6,5.0,3.1
5,6,2,D1 Sabaneta Parque,10:20,10:50,0.6,4.4,0.1
6,7,1,Éxito Mayorca,10:59,11:29,1.2,3.2,3.4
7,8,9,Ara Envigado El Dorado,11:36,12:06,0.6,2.6,2.6
8,9,8,D1 Envigado La Magnolia,12:10,12:40,0.7,1.9,1.6
9,10,11,Carulla Poblado,12:52,13:22,1.1,0.8,4.9


## 8. Verificación de restricciones

`verificar_solucion` recalcula todo desde la **matriz original**: visitas, inicio y fin en el depot, carga,
duración, flota y costo, respetando el sentido de cada tramo. La aplicación ejecuta esta misma función
antes de mostrar cualquier resultado.

In [17]:
filas = []
for d in resultados["dias"]:
    if d["estado"] == "resuelto":
        for v in d["verificacion"]:
            filas.append({"Día": d["dia"], "Criterio": v["criterio"], "Cumple": "✔" if v["cumple"] else "✘", "Detalle": v["detalle"]})
verif = pd.DataFrame(filas)
print("Todas las restricciones se cumplen:", all(d.get("verificacion_ok", True) for d in resultados["dias"]))
verif

Todas las restricciones se cumplen: True


,Día,Criterio,Cumple,Detalle
0,Lunes,Cada cliente con demanda se visita exactamente...,✔,30 de 30 atendidos
1,Lunes,Toda ruta inicia y termina en el depot (nodo 0),✔,5 rutas: 0 → … → 0
2,Lunes,Carga ≤ capacidad en cada ruta,✔,"máxima utilización 100,0 %"
3,Lunes,Duración ≤ 480 min en cada ruta (recalculada),✔,"ruta más larga 434,8 min"
4,Lunes,Vehículos usados ≤ disponibles por tipo,✔,A: 1/1 · B: 2/2 · C: 2/2
5,Lunes,Costo del tablero = costo recalculado desde la...,✔,tablero $ 493.000 · recalculado $ 493.000
6,Martes,Cada cliente con demanda se visita exactamente...,✔,30 de 30 atendidos
7,Martes,Toda ruta inicia y termina en el depot (nodo 0),✔,5 rutas: 0 → … → 0
8,Martes,Carga ≤ capacidad en cada ruta,✔,"máxima utilización 100,0 %"
9,Martes,Duración ≤ 480 min en cada ruta (recalculada),✔,"ruta más larga 431,0 min"


## 9. Días infactibles: diagnóstico

Con el archivo de prueba `d_dia_infactible.xlsx`, donde el Viernes suma 30,5 t contra 28 t de flota,
el día se informa con su causa y los demás se resuelven normalmente. Para no repetir 30 s de cálculo por día,
esta celda usa un tiempo límite de 5 s.

In [18]:
res_d = ruteo.resolver_archivo("datos/pruebas/d_dia_infactible.xlsx", parametros=ruteo.parametros_por_defecto(tiempo_limite_s=5))
print(res_d["mensaje_final"])
for d in res_d["dias"]:
    if d["estado"] == "infactible":
        print(f"{d['dia']}: {d['diagnostico']['titulo']} — {d['diagnostico']['mensaje']}")

4 día(s) resuelto(s) · 1 sin datos · 1 infactible(s)
Viernes: Demanda mayor que la capacidad de la flota — La demanda del día es 30,5 t y la flota completa carga 28,0 t: faltan 2,5 t (8,9 %). Se necesita más flota o reprogramar pedidos.


Resultados guardados para comparar con la aplicación, que debe dar los mismos costos para el mismo archivo:

In [19]:
Path("salidas").mkdir(exist_ok=True)
Path("salidas/resultados_notebook.json").write_text(json.dumps(resultados, ensure_ascii=False, indent=1), encoding="utf-8")
{d["dia"]: round(d["indicadores"]["costo_total"]) for d in resultados["dias"] if d["estado"] == "resuelto"}

{'Lunes': 493000, 'Martes': 480460, 'Miércoles': 383180, 'Jueves': 464530}

## 10. Cómo iniciar la aplicación

Desde una terminal, en la carpeta del proyecto:

```bash
pip install -r requirements.txt
python app.py
```

El navegador se abre en **http://127.0.0.1:8000** con el archivo de ejemplo ya resuelto.
Para resolver otros días, por ejemplo Viernes y Sábado, llene las columnas en Excel y use **Cargar archivo de Excel**.
No hay que tocar código.